<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Memory-Systeme
</b></font> </br></p>

---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M16-Memory-Systeme"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.10
langchain-chroma                         1.1.0
langchain-classic                        1.0.1
langchain-community                      0.4.1
langchain-core                           1.2.16
langchain-ollama                         1.0.1
langchain-openai                         1.1.10
langchain-text-splitters                 1.1.1
langgraph                                1.0.9
langgraph-checkpoint                     4.0.0
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.9

IP-Adresse: 136.112.206.57
Hostname: 57.206.112.136.bc.googleusercontent.com
Stadt: Council Bluffs
Region: Iowa
Land: US
Koordinaten: 41.2619,-95.8608
Provider: AS396982 Google LLC
Postleitzahl: 51502
Zeitzone: America/Chicago


In [2]:
#@title 📦 Pakete installieren{ display-mode: "form" }
install_packages([("langgraph-checkpoint-sqlite", "langgraph.checkpoint.sqlite")])

🔄 Installiere langgraph-checkpoint-sqlite...
✅ langgraph-checkpoint-sqlite erfolgreich installiert und importiert


<p><font color='black' size="5">
Lernziele
</font></p>


Nach diesem Modul kannst du:

- **Kurzzeit-Memory** einsetzen: Conversation Buffer, Sliding Window, Summarization
- **Langzeit-Memory** aufbauen: Semantisches Memory mit ChromaDB (`persist_directory`) und `SqliteSaver`
- **Entity Memory** nutzen: Key-Value-Struktur für Entitäten im Graph-State
- **Per-User-Memory** mit Thread-IDs realisieren
- Kurzzeit- und Langzeit-Strategien gezielt kombinieren

---

## Warum brauchen Agenten Memory?

Ein LLM hat kein eingebautes Gedächtnis. Ohne explizite Mechanismen beginnt jedes Gespräch von vorne.

| Problem | Lösung |
|---------|--------|
| Gesprächsverlauf geht verloren | Kurzzeit-Memory im State |
| Kontext wächst unbegrenzt | Sliding Window oder Summarization |
| Nutzerdaten nach Session weg | Langzeit-Memory (Vektordatenbank) |
| Kein Personalisierungspotenzial | Entity Memory oder Per-User-Store |

In [3]:
import os
import operator
from typing import TypedDict, Annotated, Optional

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain.chat_models import init_chat_model
from langchain_core.messages import trim_messages, RemoveMessage, SystemMessage, HumanMessage, AIMessage

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

In [4]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TB
    MEMORY[Memory-Systeme] --> SHORT[Kurzzeit-Memory]
    MEMORY --> LONG[Langzeit-Memory]

    SHORT --> BUFFER["Conversation Buffer<br/>Voller Verlauf"]
    SHORT --> WINDOW["Sliding Window<br/>Letzte N Nachrichten"]
    SHORT --> SUMMAR["Summarization<br/>Komprimierter Verlauf"]

    LONG --> SEMANTIC["Semantisches Memory<br/>Vektordatenbank"]
    LONG --> ENTITY["Entity Memory<br/>Key-Value Struktur"]

    style MEMORY fill:#87CEEB,color:#000
    style SHORT  fill:#90EE90,color:#000
    style LONG   fill:#90EE90,color:#000
'''

mermaid(diagram, width=1050)

---
## 1 Kurzzeit-Memory

Drei Strategien, die sich darin unterscheiden, wie viel Verlauf aufbewahrt wird.

### 1.1 Conversation Buffer – Vollständiger Verlauf

In [5]:
class BufferState(TypedDict):
    messages: Annotated[list, add_messages]

def buffer_chat(state: BufferState) -> BufferState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(BufferState)
graph.add_node("chat", buffer_chat)
graph.add_edge(START, "chat")
graph.add_edge("chat", END)

buffer_app = graph.compile(checkpointer=MemorySaver())
mprint("✅ Conversation Buffer aufgebaut")

✅ Conversation Buffer aufgebaut

In [6]:
#@title
#@markdown   <p><font size="4" color='green'>Graph: Conversation Buffer</font> </br></p>

diagram = buffer_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

In [7]:
config = {"configurable": {"thread_id": "buffer-demo"}}

for nachricht in ["Mein Name ist Emma.", "Ich arbeite als Datenwissenschaftlerin.", "Wie heiße ich?"]:
    result = buffer_app.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config=config
    )
    antwort = result["messages"][-1].content
    mprint(f"**Nutzer:** {nachricht}  \n**Agent:** {antwort}")
    mprint("---")

**Nutzer:** Mein Name ist Emma.  
**Agent:** Hallo Emma! Wie kann ich dir helfen?

---

**Nutzer:** Ich arbeite als Datenwissenschaftlerin.  
**Agent:** Das klingt spannend, Emma! Datenwissenschaft ist ein faszinierendes Feld. An welchen Projekten arbeitest du gerade oder was interessiert dich besonders an deiner Arbeit?

---

**Nutzer:** Wie heiße ich?  
**Agent:** Du hast gesagt, dass dein Name Emma ist.

---

### 1.2 Sliding Window – Letzte N Nachrichten

`trim_messages()` begrenzt den Kontext auf die jüngsten Nachrichten, sobald ein Token-Limit erreicht wird.

In [8]:
class WindowState(TypedDict):
    messages: Annotated[list, add_messages]

def window_chat(state: WindowState) -> WindowState:
    trimmed = trim_messages(
        state["messages"],
        max_tokens=2000,
        strategy="last",
        token_counter=llm,
        include_system=True,
        allow_partial=False,
    )
    response = llm.invoke(trimmed)
    return {"messages": [response]}

graph2 = StateGraph(WindowState)
graph2.add_node("chat", window_chat)
graph2.add_edge(START, "chat")
graph2.add_edge("chat", END)
window_app = graph2.compile(checkpointer=MemorySaver())

# Veranschaulichung: trim_messages kuerzt
nachrichten = [HumanMessage(content=f"Nachricht {i}") for i in range(1, 8)]
gekuerzt = trim_messages(nachrichten, max_tokens=100, strategy="last", token_counter=llm)
mprint(f"Original: {len(nachrichten)} Nachrichten → Nach trim_messages: {len(gekuerzt)} Nachrichten")

Original: 7 Nachrichten → Nach trim_messages: 7 Nachrichten

In [9]:
#@title
#@markdown   <p><font size="4" color='green'>Graph: Sliding Window</font> </br></p>

diagram = window_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

### 1.3 Summarization Memory – Komprimierter Verlauf

Ältere Nachrichten werden vom LLM zusammengefasst statt verworfen. Der Kontext bleibt erhalten, der Token-Verbrauch wird begrenzt.

In [10]:
class SummaryState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str

def summarize_if_needed(state: SummaryState) -> SummaryState:
    # Komprimiert, sobald mehr als 6 Nachrichten vorhanden sind
    messages = state["messages"]
    if len(messages) < 6:
        return {}

    existing = state.get("summary", "")
    behalten = messages[-4:]
    zum_komprimieren = messages[:-4]

    prompt = (
        f"Fasse dieses Gespräch in 2-3 Sätzen zusammen.\n"
        f"Bisherige Zusammenfassung: {existing or 'Keine'}\n\n"
        + "\n".join(f"{m.type}: {m.content}" for m in zum_komprimieren)
    )
    neue_zusammenfassung = llm.invoke(prompt).content
    zu_entfernen = [RemoveMessage(id=m.id) for m in zum_komprimieren]
    summary_msg = SystemMessage(
        content=f"Bisheriger Verlauf (komprimiert): {neue_zusammenfassung}"
    )
    return {"messages": [summary_msg] + zu_entfernen, "summary": neue_zusammenfassung}

def summary_chat(state: SummaryState) -> SummaryState:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

graph3 = StateGraph(SummaryState)
graph3.add_node("summarize", summarize_if_needed)
graph3.add_node("chat", summary_chat)
graph3.add_edge(START, "summarize")
graph3.add_edge("summarize", "chat")
graph3.add_edge("chat", END)
summary_app = graph3.compile(checkpointer=MemorySaver())
mprint("✅ Summarization Memory aufgebaut")

✅ Summarization Memory aufgebaut

In [11]:
#@title
#@markdown   <p><font size="4" color='green'>Graph: Summarization Memory</font> </br></p>

diagram = summary_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

In [12]:
config = {"configurable": {"thread_id": "summary-demo"}}
themen = [
    "Mein Name ist Max, ich bin Softwareentwickler.",
    "Ich arbeite bei einer Berliner KI-Firma.",
    "Mein Lieblingswerkzeug ist Python.",
    "Ich interessiere mich für LangChain und LangGraph.",
    "Was weißt du über mich?",
]
for nachricht in themen:
    result = summary_app.invoke(
        {"messages": [HumanMessage(content=nachricht)]},
        config=config
    )
    antwort = result["messages"][-1].content
    n = len(result["messages"])
    mprint(f"**({n} Msgs) Nutzer:** {nachricht}  \n**Agent:** {antwort[:140]}...")
    mprint("---")

**(2 Msgs) Nutzer:** Mein Name ist Max, ich bin Softwareentwickler.  
**Agent:** Hallo Max! Schön, dich kennenzulernen. Was für Projekte oder Technologien interessieren dich besonders in der Softwareentwicklung?...

---

**(4 Msgs) Nutzer:** Ich arbeite bei einer Berliner KI-Firma.  
**Agent:** Das klingt spannend! Berlin hat eine lebendige Tech- und KI-Szene. An welchen Projekten arbeitest du dort? Und welche Technologien oder Ansä...

---

**(6 Msgs) Nutzer:** Mein Lieblingswerkzeug ist Python.  
**Agent:** Python ist eine großartige Wahl, besonders in der KI-Entwicklung! Es bietet viele leistungsstarke Bibliotheken wie TensorFlow, PyTorch, scik...

---

**(6 Msgs) Nutzer:** Ich interessiere mich für LangChain und LangGraph.  
**Agent:** LangChain und LangGraph sind spannende Tools! LangChain ist besonders nützlich für die Entwicklung von Anwendungen, die mit Sprachmodellen i...

---

**(6 Msgs) Nutzer:** Was weißt du über mich?  
**Agent:** Ich weiß, dass du ein Softwareentwickler aus Berlin bist und in der KI-Branche arbeitest. Python ist dein bevorzugtes Werkzeug, und du inter...

---

---
## 2 Langzeit-Memory

Langzeit-Memory überlebt das Sitzungsende und steht in zukünftigen Gesprächen zur Verfügung.

### 2.1 Semantisches Memory – Vektordatenbank

In [13]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.tools import tool
import os

# persist_directory sorgt dafür, dass der Vektorspeicher
# einen Notebook-Neustart überlebt (Langzeit-Persistenz)
MEMORY_DIR = "./semantic_memory"
os.makedirs(MEMORY_DIR, exist_ok=True)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
memory_store = Chroma(
    collection_name="agent_memory",
    embedding_function=embeddings,
    persist_directory=MEMORY_DIR,   # ← persistente Speicherung auf Disk
)

@tool
def memory_speichern(information: str) -> str:
    '''MEMORY SPEICHERN - Speichert eine wichtige Information dauerhaft.

    Verwende dieses Tool wenn der Nutzer Relevantes erwähnt:
    - Persönliche Fakten ("Ich arbeite als Arzt")
    - Präferenzen ("Ich mag kurze Antworten")
    - Ziele ("Ich lerne Python fuer Data Science")

    Args:
        information: Die Information in einem präzisen Satz

    Returns:
        Bestätigung der Speicherung
    '''
    memory_store.add_texts([information])
    return f"Gespeichert: {information}"

@tool
def memory_abrufen(frage: str) -> str:
    '''MEMORY ABRUFEN - Durchsucht das Gedächtnis nach relevanten Informationen.

    Args:
        frage: Suchanfrage in natürlicher Sprache

    Returns:
        Relevante gespeicherte Informationen
    '''
    docs = memory_store.similarity_search(frage, k=3)
    if not docs:
        return "Keine relevanten Informationen gefunden."
    return "\n".join(f"- {d.page_content}" for d in docs)

mprint(f"✅ Semantisches Memory initialisiert (persistent: {MEMORY_DIR})")

✅ Semantisches Memory initialisiert (persistent: ./semantic_memory)

In [14]:
from langchain.agents import create_agent

memory_agent = create_agent(
    model=llm,
    tools=[memory_speichern, memory_abrufen],
    system_prompt=(
        "Du bist ein hilfreicher Assistent mit Langzeitgedächtnis. "
        "Speichere wichtige Informationen über den Nutzer und rufe sie bei Bedarf ab."
    )
)

result1 = memory_agent.invoke({
    "messages": [HumanMessage(content="Ich heiße Lena und lerne KI-Programmierung.")]
})
mprint(f"**Agent:** {result1['messages'][-1].content}")

result2 = memory_agent.invoke({
    "messages": [HumanMessage(content="Was weißt du über mich?")]
})
mprint(f"**Agent:** {result2['messages'][-1].content}")

**Agent:** Ich habe die Informationen gespeichert: Du heißt Lena und lernst KI-Programmierung. Wie kann ich dir weiterhelfen?

**Agent:** Ich weiß, dass du Lena heißt und dass du KI-Programmierung lernst. Wenn es noch mehr gibt, das du teilen möchtest, lass es mich wissen!

### 2.2 Entity Memory – Key-Value für Entitäten

In [15]:
from pydantic import BaseModel, Field

class Entitaet(BaseModel):
    name: str = Field(description="Name der Entität (Person, Ort, Projekt)")
    beschreibung: str = Field(description="Kurze Beschreibung in einem Satz")

class EntitaetListe(BaseModel):
    entitaeten: list[Entitaet] = Field(default_factory=list)

class EntityState(TypedDict):
    messages: Annotated[list, add_messages]
    entity_memory: dict

extractor = llm.with_structured_output(EntitaetListe)

# Frage-Präfixe, bei denen keine Entitäten extrahiert werden sollen
FRAGE_PRAEFIXE = ("was ", "wer ", "wie ", "wo ", "wann ", "warum ", "welche", "kennst")

def entity_extraktor(state: EntityState) -> EntityState:
    letzte = state["messages"][-1].content.strip()

    # Bug-Fix 1: Fragen überspringen – sie enthalten keine neuen Fakten
    if letzte.endswith("?") or letzte.lower().startswith(FRAGE_PRAEFIXE):
        return {}

    result = extractor.invoke(
        f"Extrahiere wichtige Entitäten (Personen, Projekte, Orte) aus:\n{letzte}"
    )
    updated = dict(state.get("entity_memory", {}))
    for e in result.entitaeten:
        # Bug-Fix 2: Merge statt überschreiben
        if e.name in updated and e.beschreibung not in updated[e.name]:
            updated[e.name] = updated[e.name] + "; " + e.beschreibung
        else:
            updated[e.name] = e.beschreibung
    return {"entity_memory": updated}

def entity_chat(state: EntityState) -> EntityState:
    kontext = "\n".join(f"- {k}: {v}" for k, v in state.get("entity_memory", {}).items())
    system = f"Du kennst folgende Entitäten:\n{kontext}" if kontext else ""
    msgs = ([SystemMessage(content=system)] if system else []) + list(state["messages"])
    return {"messages": [llm.invoke(msgs)]}

graph4 = StateGraph(EntityState)
graph4.add_node("extraktor", entity_extraktor)
graph4.add_node("chat", entity_chat)
graph4.add_edge(START, "extraktor")
graph4.add_edge("extraktor", "chat")
graph4.add_edge("chat", END)
entity_app = graph4.compile(checkpointer=MemorySaver())
mprint("✅ Entity Memory aufgebaut")

✅ Entity Memory aufgebaut

In [16]:
#@title
#@markdown   <p><font size="4" color='green'>Graph: Entity Memory</font> </br></p>

diagram = entity_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

In [17]:
config = {"configurable": {"thread_id": "entity-demo"}}
nachrichten = [
    "Ich arbeite mit Tom an Projekt Alpha.",
    "Tom ist unser Tech Lead und Projekt Alpha ist eine KI-Plattform.",
    "Was weißt du über Tom?",
]
for msg in nachrichten:
    result = entity_app.invoke(
        {"messages": [HumanMessage(content=msg)]},
        config=config
    )
    mem = result.get("entity_memory", {})
    antwort = result["messages"][-1].content
    mprint(f"**Nutzer:** {msg}")
    if mem:
        mprint(f"*Entity Memory: {mem}*")
    mprint(f"**Agent:** {antwort}")
    mprint("---")

**Nutzer:** Ich arbeite mit Tom an Projekt Alpha.

*Entity Memory: {'Tom': 'Eine Person, die an Projekt Alpha arbeitet.', 'Projekt Alpha': 'Ein Projekt, an dem Tom und ich arbeiten.'}*

**Agent:** Das klingt spannend! Was genau macht ihr an Projekt Alpha?

---

**Nutzer:** Tom ist unser Tech Lead und Projekt Alpha ist eine KI-Plattform.

...

---

**Nutzer:** Was weißt du über Tom?

*Entity Memory: {'Tom': 'Eine Person, die an Projekt Alpha arbeitet.; Tech Lead des Projekts.', 'Projekt Alpha': 'Ein Projekt, an dem Tom und ich arbeiten.; Eine KI-Plattform.'}*

**Agent:** Tom ist eine Person, die an Projekt Alpha arbeitet und die Rolle des Tech Leads innehat. Er spielt eine wichtige Rolle in der Entwicklung der KI-Plattform, an der ihr gemeinsam arbeitet. Wenn du mehr über seine spezifischen Fähigkeiten oder Erfahrungen wissen möchtest, müsstest du mir mehr Informationen geben.

---

### 2.3 Persistenter Checkpointer – SqliteSaver

**Problem:** `MemorySaver` speichert den Graphzustand nur im RAM – nach einem Notebook-Neustart ist der Zustand weg.

**Lösung:** `SqliteSaver` schreibt jeden Checkpoint in eine SQLite-Datei. Der Graph-Zustand überlebt Neustarts und ist für alle Threads dauerhaft verfügbar.

| | `MemorySaver` | `SqliteSaver` |
|--|--------------|---------------|
| **Speicher** | RAM | SQLite-Datei auf Disk |
| **Neustart** | ❌ Zustand verloren | ✅ Zustand erhalten |
| **Einsatz** | Demos, Entwicklung | Staging, Production |
| **Setup** | `MemorySaver()` | `SqliteSaver(conn)` |

In [18]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3

# Persistente SQLite-Verbindung (Datei bleibt nach Neustart erhalten)
conn = sqlite3.connect("./agent_sessions.db", check_same_thread=False)
persistent_checkpointer = SqliteSaver(conn)

# Gleicher Graph wie Conversation Buffer, jetzt mit SqliteSaver
persistent_graph = StateGraph(BufferState)
persistent_graph.add_node("chat", buffer_chat)
persistent_graph.add_edge(START, "chat")
persistent_graph.add_edge("chat", END)
persistent_app = persistent_graph.compile(checkpointer=persistent_checkpointer)
mprint("✅ Persistenter Graph mit SqliteSaver aufgebaut (agent_sessions.db)")

✅ Persistenter Graph mit SqliteSaver aufgebaut (agent_sessions.db)

In [19]:
config_p = {"configurable": {"thread_id": "session-clara"}}

# Runde 1 – Information speichern
result = persistent_app.invoke(
    {"messages": [HumanMessage(content="Mein Name ist Clara und ich lerne LangGraph.")]},
    config=config_p
)
mprint(f"**Agent (Runde 1):** {result['messages'][-1].content}")

# Runde 2 – Memory-Recall (Zustand aus SQLite)
result = persistent_app.invoke(
    {"messages": [HumanMessage(content="Was weißt du über mich?")]},
    config=config_p
)
mprint(f"**Agent (Runde 2):** {result['messages'][-1].content}")

mprint("\n*Zustand gespeichert in agent_sessions.db – bleibt nach Notebook-Neustart erhalten.*")

**Agent (Runde 1):** Hallo Clara! Das klingt spannend! LangGraph ist ein interessantes Tool. Was genau möchtest du darüber lernen oder wissen? Gibt es spezielle Fragen oder Themen, die dich interessieren?

**Agent (Runde 2):** Ich weiß nur, dass dein Name Clara ist und dass du LangGraph lernst. Ich habe keine weiteren Informationen über dich, da ich keine persönlichen Daten speichern oder abrufen kann. Wenn du mir mehr über dich erzählen möchtest oder Fragen hast, stehe ich dir gerne zur Verfügung!


*Zustand gespeichert in agent_sessions.db – bleibt nach Notebook-Neustart erhalten.*

In [20]:
#@title
#@markdown   <p><font size="4" color='green'>  Graph: Persistenter Checkpointer (SqliteSaver)</font> </br></p>

diagram = persistent_app.get_graph().draw_mermaid()
mermaid(diagram, width=400)

---
## 3 Per-User Memory

In Multi-User-Systemen trennt eine eindeutige `thread_id` pro Nutzer den Kontext vollständig.

In [21]:
def get_config(user_id: str, session_id: str = "default") -> dict:
    # Erstellt nutzerspezifische Konfiguration
    return {"configurable": {"thread_id": f"{user_id}:{session_id}"}}

config_alice = get_config("alice")
config_bob   = get_config("bob")

buffer_app.invoke(
    {"messages": [HumanMessage(content="Ich heiße Alice und mag Python.")]},
    config=config_alice
)
buffer_app.invoke(
    {"messages": [HumanMessage(content="Ich heiße Bob und arbeite mit Java.")]},
    config=config_bob
)

# Beide fragen nach ihrem Namen - vollstaendig getrennte Kontexte
for name, cfg in [("Alice", config_alice), ("Bob", config_bob)]:
    result = buffer_app.invoke(
        {"messages": [HumanMessage(content="Wie heiße ich?")]},
        config=cfg
    )
    mprint(f"**{name} fragt:** Wie heiße ich?  \n**Agent:** {result['messages'][-1].content}")

**Alice fragt:** Wie heiße ich?  
**Agent:** Du hast gesagt, dass du Alice heißt.

**Bob fragt:** Wie heiße ich?  
**Agent:** Du hast gesagt, dass du Bob heißt. Gibt es etwas Bestimmtes, worüber du sprechen möchtest?

---
## Zusammenfassung

| Memory-Typ | Implementierung | Persistenz | Einsatz |
|-----------|----------------|-----------|---------|
| **Conversation Buffer** | `add_messages` Reducer | ❌ RAM | Kurze Gespräche, einfache Demos |
| **Sliding Window** | `trim_messages()` | ❌ RAM | Lange Gespräche, Token-Budget begrenzen |
| **Summarization** | `RemoveMessage` + LLM | ❌ RAM | Multi-Turn mit wichtigem Kontext |
| **Semantisches Memory** | ChromaDB + `persist_directory` | ✅ Disk | Nutzerpräferenzen über Sessions |
| **Entity Memory** | `with_structured_output` + Dict | ❌ RAM | CRM-Agenten, Support-Systeme |
| **Persistenter Checkpointer** | `SqliteSaver` | ✅ Disk | Staging, Production, Multi-User |
| **Per-User Memory** | Thread-IDs im Checkpointer | je Checkpointer | Multi-User-Anwendungen |

**Faustregel:**
- **Kurzzeit (Demo/Dev)** → `MemorySaver` + Conversation Buffer oder Sliding Window
- **Langzeit (Semantic)** → `ChromaDB` mit `persist_directory`
- **Langzeit (Sessions)** → `SqliteSaver` als Checkpointer
- **Per-User** → eindeutige Thread-IDs + persistenter Checkpointer

Verwandte Module: M15 (Checkpointing & Sessions), M18 (Human-in-the-Loop), M19 (Multi-Agent)

# A | Aufgaben
---



<p><font color='black' size="5">
Aufgabe 1: Sliding Window – Limit anpassen
</font></p>

**Schwierigkeit:** 1/5

Ändere in `window_chat()` den Parameter `max_tokens` von `2000` auf `500`.
Führe danach eine längere Konversation (5+ Nachrichten). Beobachte, ab wann frühere Informationen verloren gehen.

<p><font color='black' size="5">
Aufgabe 2: Summarization – Prompt verbessern
</font></p>

**Schwierigkeit:** 2/5

Verbessere den Summarization-Prompt in `summarize_if_needed()`, sodass er explizit Names, Orte und Präferenzen extrahiert:

```python
prompt = (
    "Fasse dieses Gespräch prägnant zusammen. "
    "Extrahiere explizit: Namen, Orte, Berufe, Präferenzen.\n"
    ...
)
```

Teste, ob die Zusammenfassung danach relevanter ist.

<p><font color='black' size="5">
Aufgabe 3: Entity Memory erweitern
</font></p>

**Schwierigkeit:** 3/5

Erweitere das `Entitaet`-Modell um ein Feld `kategorie` (Person / Projekt / Ort):

```python
class Entitaet(BaseModel):
    name: str
    beschreibung: str
    kategorie: str = Field(description="Person, Projekt oder Ort")
```

Passe `entity_extraktor` an und prüfe, ob die Kategorisierung korrekt funktioniert.

<p><font color='black' size="5">
Aufgabe 4: SqliteSaver – Persistenz testen
</font></p>

**Schwierigkeit:** 3/5

1. Führe die SqliteSaver-Demo aus und speichere Informationen in einem Thread.
2. Ändere die `thread_id` auf `"session-neu"` und stelle eine Frage – der Kontext sollte fehlen.
3. Wechsle zurück zur alten `thread_id` – der Kontext sollte wieder da sein.

Warum funktioniert das auch nach einem "Neustart" (Kernel Restart + Run All)?